In [0]:
%pip install requests

In [0]:
# Notebook: 2_Silver_Transformation
# Language: Python with SQL

from pyspark.sql.functions import from_json, explode, col, to_date, lit
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# Configuration
BRONZE_TABLE = "mutual_fund_project.bronze.raw_fund_data"
SILVER_TABLE = "mutual_fund_project.silver.fund_nav_details"

# 1. Define the schema to parse the nested JSON string
# This matches the structure of the 'meta' and 'data' fields in the API response
json_schema = StructType([
    StructField("meta", StructType([
        StructField("fund_house", StringType()),
        StructField("scheme_name", StringType()),
        StructField("scheme_code", StringType())
    ])),
    StructField("data", ArrayType(StructType([
        StructField("date", StringType()),
        StructField("nav", StringType())
    ])))
])

In [0]:
# 2. Read the raw data from the Bronze table
bronze_df = spark.read.table(BRONZE_TABLE)

# 3. Parse, flatten, and clean the data
silver_df = (
    bronze_df
    # Parse the raw_data JSON string using the defined schema
    .withColumn("parsed_data", from_json(col("raw_data"), json_schema))
    
    # Select the required fields from the parsed data
    .select(
        col("parsed_data.meta.fund_house").alias("fund_house"),
        col("parsed_data.meta.scheme_name").alias("scheme_name"),
        col("parsed_data.meta.scheme_code").alias("scheme_code"),
        # Explode the 'data' array to create one row per NAV entry
        explode(col("parsed_data.data")).alias("nav_data")
    )
    
    # Flatten the struct from the exploded array
    .select(
        "fund_house",
        "scheme_name",
        "scheme_code",
        to_date(col("nav_data.date"), "dd-MM-yyyy").alias("date"),
        col("nav_data.nav").cast("float").alias("nav")
    )
    # Filter out any rows where parsing might have failed
    .filter(col("date").isNotNull() & col("nav").isNotNull())
)

In [0]:
# 4. Create the target Silver table if it doesn't exist
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
        fund_house STRING,
        scheme_name STRING,
        scheme_code STRING,
        date DATE,
        nav FLOAT
    )
    USING DELTA
    PARTITIONED BY (fund_house)
""")

In [0]:
# 5. Perform an incremental load using a MERGE statement
# This ensures we only add new records or update existing ones, making the process idempotent.
silver_df.createOrReplaceTempView("silver_updates")

spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS target
    USING silver_updates AS source
    ON target.scheme_code = source.scheme_code AND target.date = source.date
    WHEN MATCHED THEN
        UPDATE SET target.nav = source.nav
    WHEN NOT MATCHED THEN
        INSERT (fund_house, scheme_name, scheme_code, date, nav)
        VALUES (source.fund_house, source.scheme_name, source.scheme_code, source.date, source.nav)
""")

print(f"Successfully merged data into {SILVER_TABLE}.")